In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [17]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate

In [5]:
pdfLoader = PyPDFLoader("../data/data_science_syllabus.pdf")
docs = pdfLoader.load()
len(docs)

10

In [8]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
split_docs = splitter.split_documents(docs)
len(split_docs)

11

In [11]:
embedding_model = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")
vector_store = Chroma.from_documents(
    documents = split_docs, 
    embedding = embedding_model
)

In [19]:
query = "Machine Learning and Data Science content"
response = vector_store.similarity_search(query)
len(response)

4

In [20]:
context = ""
for res in response:
    context += res.page_content + "\n"

context

'🎤  Mock Interview: Stats Scenarios \ue09d Probabilities \ue09d Tests\n✅  Module 6: Machine Learning – I (Supervised \nLearning) (4 weeks)\nDuration: Month 6\nTopics:\nML pipeline\nRegression: Linear, Logistic\nDecision Tree, Random Forest, KNN\nTrain-test split, model evaluation\nTools:\nScikit-learn, Google Colab, ChatGPT, PyCaret (optional)\nMini Project:\nLoan Approval or House Price Prediction\nPredict Diabetes from health dataset\n🎤  Mock Interview: Supervised Learning Models \ue09d Metrics\n✅  Module 7: Machine Learning – II (Unsupervised & \nFeature Engineering) (3 weeks)\nDuration: Month 7\nTopics:\nKMeans Clustering\nDimensionality Reduction: PCA\nFeature selection, encoding, scaling\nModel tuning \ue081GridSearchCV\ue082\nTools:\nScikit-learn, Seaborn, Colab\nMini Project:\n📘  1\ue088 Y e ar R o admap: Dat a Anal y tics, Dat a Science & GenAI\n5\n🎤  Mock Interview: Stats Scenarios \ue09d Probabilities \ue09d Tests\n✅  Module 6: Machine Learning – I (Supervised \nLearning) (4

In [21]:
def get_context(query):
    response = vector_store.similarity_search(query)
    context = ""
    for res in response:
        context += res.page_content + "\n"
    return {"context": context, "question": query}

In [18]:
model = ChatGroq(model="openai/gpt-oss-20b")

In [22]:
prompt_template = ChatPromptTemplate.from_template(
    """
        You are a helpful assistant and answer questions provided, based on the provided context. If you don't know the answer, say you don't know.
        Context: {context}
        Question: {question}
    """
)

In [23]:
rag_chain = get_context | prompt_template | model

In [26]:
response = rag_chain.invoke("What is duration of course?")
print(response.content)

The course is structured as a **4‑week program (Month 1)**, with a brief orientation of 1–2 days at the start.
